# Week 4 — Day 1: Train, Validation & Test Splits
## Breast Cancer Classification Dataset

### Objective

This notebook demonstrates the correct use of training, validation,
and test sets for a breast cancer classification problem.

We will:

1. Load and inspect the dataset.
2. Prepare the features and target.
3. Create a 60/20/20 train/validation/test split.
4. Preserve class proportions using stratification.
5. Train a classification model.
6. Tune one hyperparameter using the validation set only.
7. Evaluate the final model on the test set exactly once.
8. Explain why tuning against the test set causes data leakage.
9. Explain why a single validation split can still be misleading.

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [7]:
df = pd.read_csv("BreastCancer.csv")

df.head()

,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0


In [8]:
print("Dataset shape:", df.shape)

Dataset shape: (699, 11)


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Id               699 non-null    int64  
 1   Cl.thickness     699 non-null    int64  
 2   Cell.size        699 non-null    int64  
 3   Cell.shape       699 non-null    int64  
 4   Marg.adhesion    699 non-null    int64  
 5   Epith.c.size     699 non-null    int64  
 6   Bare.nuclei      683 non-null    float64
 7   Bl.cromatin      699 non-null    int64  
 8   Normal.nucleoli  699 non-null    int64  
 9   Mitoses          699 non-null    int64  
 10  Class            699 non-null    int64  
dtypes: float64(1), int64(10)
memory usage: 60.2 KB


In [10]:
df.describe()

,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
count,6.990000e+02,699.000000,699.000000,699.000000,699.000000,699.000000,683.000000,699.000000,699.000000,699.000000,699.000000
mean,1.071704e+06,4.417740,3.134478,3.207439,2.806867,3.216023,3.544656,3.437768,2.866953,1.589413,0.344778
std,6.170957e+05,2.815741,3.051459,2.971913,2.855379,2.214300,3.643857,2.438364,3.053634,1.715078,0.475636
min,6.163400e+04,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,8.706885e+05,2.000000,1.000000,1.000000,1.000000,2.000000,1.000000,2.000000,1.000000,1.000000,0.000000
50%,1.171710e+06,4.000000,1.000000,1.000000,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000,0.000000
75%,1.238298e+06,6.000000,5.000000,5.000000,4.000000,4.000000,6.000000,5.000000,4.000000,1.000000,1.000000
max,1.345435e+07,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,1.000000


In [11]:
df.isnull().sum()

Id                  0
Cl.thickness        0
Cell.size           0
Cell.shape          0
Marg.adhesion       0
Epith.c.size        0
Bare.nuclei        16
Bl.cromatin         0
Normal.nucleoli     0
Mitoses             0
Class               0
dtype: int64

In [12]:
print(df["Bare.nuclei"].isnull().sum())

16


In [13]:
print(df["Class"].value_counts())

Class
0    458
1    241
Name: count, dtype: int64


In [14]:
print(df["Class"].value_counts(normalize=True))

Class
0    0.655222
1    0.344778
Name: proportion, dtype: float64


In [15]:
df["Bare.nuclei"] = df["Bare.nuclei"].fillna(
    df["Bare.nuclei"].median()
)

In [16]:
df.isnull().sum()

Id                 0
Cl.thickness       0
Cell.size          0
Cell.shape         0
Marg.adhesion      0
Epith.c.size       0
Bare.nuclei        0
Bl.cromatin        0
Normal.nucleoli    0
Mitoses            0
Class              0
dtype: int64

In [17]:
X = df.drop(columns=["Id", "Class"])
y = df["Class"]

In [18]:
print("Features:", X.columns.tolist())
print("Target:", y.name)

Features: ['Cl.thickness', 'Cell.size', 'Cell.shape', 'Marg.adhesion', 'Epith.c.size', 'Bare.nuclei', 'Bl.cromatin', 'Normal.nucleoli', 'Mitoses']
Target: Class


## Feature and Target Selection

The `Class` column is the prediction target.

The `Id` column is removed because it is an identifier rather than a
meaningful predictive feature. Keeping an identifier can introduce
irrelevant patterns into the model.

The remaining clinical/cell-related measurements are used as input
features.

In [19]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

In [21]:
total = len(df)

print("Total samples:", total)
print(
    f"Training: {len(X_train)} "
    f"({len(X_train)/total:.2%})"
)

print(
    f"Validation: {len(X_val)} "
    f"({len(X_val)/total:.2%})"
)

print(
    f"Test: {len(X_test)} "
    f"({len(X_test)/total:.2%})"
)

Total samples: 699
Training: 419 (59.94%)
Validation: 140 (20.03%)
Test: 140 (20.03%)


In [22]:
print("Original:")
print(y.value_counts(normalize=True))

print("\nTraining:")
print(y_train.value_counts(normalize=True))

print("\nValidation:")
print(y_val.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))

Original:
Class
0    0.655222
1    0.344778
Name: proportion, dtype: float64

Training:
Class
0    0.653938
1    0.346062
Name: proportion, dtype: float64

Validation:
Class
0    0.657143
1    0.342857
Name: proportion, dtype: float64

Test:
Class
0    0.657143
1    0.342857
Name: proportion, dtype: float64


## Three-Way Split

The dataset is divided into three subsets:

| Set | Approximate Size | Purpose |
|---|---:|---|
| Training | 60% | Learn model parameters |
| Validation | 20% | Select the model configuration |
| Test | 20% | Final unbiased evaluation |

Because this is a classification problem, stratification is used to
preserve the class distribution across the three subsets.

A fixed `random_state=42` is used to make the split reproducible.

In [23]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
train_pred = model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

train_f1 = f1_score(
    y_train,
    train_pred,
    average="weighted"
)

print("Training Accuracy:", train_accuracy)
print("Training F1:", train_f1)

Training Accuracy: 1.0
Training F1: 1.0


In [25]:
val_pred = model.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_pred
)

val_f1 = f1_score(
    y_val,
    val_pred,
    average="weighted"
)

print("Validation Accuracy:", val_accuracy)
print("Validation F1:", val_f1)

Validation Accuracy: 0.9428571428571428
Validation F1: 0.9433650793650794


In [26]:
val_pred = model.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_pred
)

val_f1 = f1_score(
    y_val,
    val_pred,
    average="weighted"
)

print("Validation Accuracy:", val_accuracy)
print("Validation F1:", val_f1)

Validation Accuracy: 0.9428571428571428
Validation F1: 0.9433650793650794


In [27]:
results = []

for n in [50, 100, 200]:

    model = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    f1 = f1_score(
        y_val,
        predictions,
        average="weighted"
    )

    results.append({
        "n_estimators": n,
        "validation_accuracy": accuracy,
        "validation_f1": f1
    })

results_df = pd.DataFrame(results)

results_df

,n_estimators,validation_accuracy,validation_f1
0,50,0.942857,0.943365
1,100,0.942857,0.943365
2,200,0.942857,0.943575


In [28]:
best_row = results_df.loc[
    results_df["validation_f1"].idxmax()
]

best_n_estimators = int(
    best_row["n_estimators"]
)

best_validation_f1 = best_row["validation_f1"]

print("Best n_estimators:", best_n_estimators)
print("Best Validation F1:", best_validation_f1)

Best n_estimators: 200
Best Validation F1: 0.9435747332640921


## Hyperparameter Selection

The `n_estimators` hyperparameter was tuned using only the validation
set.

The test set was not used to select the best value.

The configuration with the highest validation F1 score was selected
as the final configuration.

In [29]:
final_model = RandomForestClassifier(
    n_estimators=best_n_estimators,
    random_state=42
)

final_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [30]:
test_pred = final_model.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

test_f1 = f1_score(
    y_test,
    test_pred,
    average="weighted"
)

print("Final Test Accuracy:", test_accuracy)
print("Final Test F1:", test_f1)

Final Test Accuracy: 0.9642857142857143
Final Test F1: 0.9645308650833512


In [31]:
print(
    classification_report(
        y_test,
        test_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      0.96      0.97        92
           1       0.92      0.98      0.95        48

    accuracy                           0.96       140
   macro avg       0.96      0.97      0.96       140
weighted avg       0.97      0.96      0.96       140



In [32]:
final_results = pd.DataFrame({
    "Metric": [
        "Validation F1",
        "Test F1",
        "Test Accuracy"
    ],
    "Score": [
        best_validation_f1,
        test_f1,
        test_accuracy
    ]
})

final_results

,Metric,Score
0,Validation F1,0.943575
1,Test F1,0.964531
2,Test Accuracy,0.964286


## Why We Must Not Tune Against the Test Set

If we repeatedly evaluated different models or hyperparameters on the
test set and selected the configuration with the best test score, the
test data would influence our development decisions.

The model would therefore become indirectly adapted to the test set.

As a result, the reported test score would no longer be an honest
estimate of performance on completely unseen data.

The correct workflow is:

Training Set
    ↓
Train the model

Validation Set
    ↓
Select model and hyperparameters

Test Set
    ↓
Evaluate the final model once

## Why One Validation Set Can Still Mislead

The validation set is better than repeatedly tuning against the test set,
but it is still only one particular subset of the data.

On a relatively small dataset, the validation set may contain an
unusually easy or difficult collection of examples.

This can make our model selection depend partly on the randomness of
that particular split.

This limitation motivates cross-validation, which will be covered on
Day 2.